In [1]:
import pandas as pd
import numpy as np

fy2013 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/LCA_FY2013.xlsx"

df = pd.read_excel(fy2013)

print(df.shape)
print(df.head())

(442275, 35)
      LCA_CASE_NUMBER               STATUS     LCA_CASE_SUBMIT  \
0  I-200-12272-415836            WITHDRAWN 2012-09-28 13:10:13   
1  I-200-10166-534717  CERTIFIED-WITHDRAWN 2010-06-15 18:29:22   
2  I-200-12272-343338            WITHDRAWN 2012-09-28 15:54:12   
3  I-200-12089-908250  CERTIFIED-WITHDRAWN 2012-03-30 09:46:49   
4  I-200-11192-575436  CERTIFIED-WITHDRAWN 2011-07-18 15:25:58   

        Decision_Date VISA_CLASS LCA_CASE_EMPLOYMENT_START_DATE  \
0 2012-10-01 10:03:05       H-1B                     2013-02-01   
1 2012-10-01 10:08:41       H-1B                     2010-06-22   
2 2012-10-01 10:09:15       H-1B                     2012-10-20   
3 2012-10-01 10:14:57       H-1B                     2012-05-01   
4 2012-10-01 10:26:36       H-1B                     2011-08-17   

  LCA_CASE_EMPLOYMENT_END_DATE  \
0          2016-01-31 00:00:00   
1          2013-01-31 00:00:00   
2          2015-10-20 00:00:00   
3          2013-04-30 00:00:00   
4          2014-0

In [3]:
print(df.columns.tolist())

['LCA_CASE_NUMBER', 'STATUS', 'LCA_CASE_SUBMIT', 'Decision_Date', 'VISA_CLASS', 'LCA_CASE_EMPLOYMENT_START_DATE', 'LCA_CASE_EMPLOYMENT_END_DATE', 'LCA_CASE_EMPLOYER_NAME', 'LCA_CASE_EMPLOYER_ADDRESS', 'LCA_CASE_EMPLOYER_CITY', 'LCA_CASE_EMPLOYER_STATE', 'LCA_CASE_EMPLOYER_POSTAL_CODE', 'LCA_CASE_SOC_CODE', 'LCA_CASE_SOC_NAME', 'LCA_CASE_JOB_TITLE', 'LCA_CASE_WAGE_RATE_FROM', 'LCA_CASE_WAGE_RATE_TO', 'LCA_CASE_WAGE_RATE_UNIT', 'FULL_TIME_POS', 'TOTAL_WORKERS', 'LCA_CASE_WORKLOC1_CITY', 'LCA_CASE_WORKLOC1_STATE', 'PW_1', 'PW_UNIT_1', 'PW_SOURCE_1', 'OTHER_WAGE_SOURCE_1', 'YR_SOURCE_PUB_1', 'LCA_CASE_WORKLOC2_CITY', 'LCA_CASE_WORKLOC2_STATE', 'PW_2', 'PW_UNIT_2', 'PW_SOURCE_2', 'OTHER_WAGE_SOURCE_2', 'YR_SOURCE_PUB_2', 'LCA_CASE_NAICS_CODE']


In [5]:
import pandas as pd

df = pd.read_excel(fy2013)

df["LCA_CASE_EMPLOYER_NAME"] = (
    df["LCA_CASE_EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["LCA_CASE_EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "LCA_CASE_EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

LCA_CASE_EMPLOYER_NAME
MICROSOFT CORPORATION                                 3902
GOOGLE INC.                                           2167
AMAZON CORPORATE LLC                                  1206
APPLE INC.                                             970
FACEBOOK, INC.                                         582
ADVANCED MICRO DEVICES, INC.                           306
MICRON TECHNOLOGY, INC.                                259
NVIDIA CORPORATION                                     238
TESLA MOTORS, INC.                                      84
APPLEXUS TECHNOLOGIES LLC                               63
APPLEXUS TECHNOLOGIES LLC.                              45
FACEBOOK SERVICES, INC.                                 43
APPLE INC                                               39
APPLE, INC.                                             31
AMAZON FULFILLMENT SERVICES, INC.                       18
AMAZON WEB SERVICES LLC                                 17
AMAZON CORPORATE, LLC            

In [7]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["LCA_CASE_EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Microsoft    3902
Apple         970
Meta          582
AMD           306
Micron        259
NVIDIA        238
Tesla          84
Name: count, dtype: int64


In [9]:
df_clean = df.copy()

In [11]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["LCA_CASE_WORKLOC1_CITY"] = (
    df["LCA_CASE_WORKLOC1_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["LCA_CASE_WORKLOC1_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Austin      187
New York     54
San Jose     25
Seattle      21
Name: count, dtype: int64


In [13]:
print(df["LCA_CASE_WAGE_RATE_UNIT"].value_counts())

LCA_CASE_WAGE_RATE_UNIT
Year    287
Name: count, dtype: int64


In [15]:
import pandas as pd

# Convert wage values to numbers
df["LCA_CASE_WAGE_RATE_FROM"] = pd.to_numeric(
    df["LCA_CASE_WAGE_RATE_FROM"],
    errors="coerce"
)

# Standardize wage units
df["LCA_CASE_WAGE_RATE_UNIT"] = (
    df["LCA_CASE_WAGE_RATE_UNIT"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["LCA_CASE_WAGE_RATE_FROM"]
    * df["LCA_CASE_WAGE_RATE_UNIT"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]].head(10))

         Company      City  LCA_CASE_WAGE_RATE_FROM LCA_CASE_WAGE_RATE_UNIT  \
146          AMD    Austin                  74464.0                    YEAR   
997         Meta  New York                 120000.0                    YEAR   
4666   Microsoft  New York                  87380.0                    YEAR   
5248         AMD    Austin                  90657.0                    YEAR   
5541         AMD    Austin                  85925.0                    YEAR   
5549         AMD    Austin                  84406.0                    YEAR   
5630   Microsoft  New York                  93766.0                    YEAR   
5933        Meta  New York                 110000.0                    YEAR   
10495        AMD    Austin                  90000.0                    YEAR   
13936        AMD    Austin                  77106.0                    YEAR   

       Annual_Wage  
146        74464.0  
997       120000.0  
4666       87380.0  
5248       90657.0  
5541       85925.0  
5549

In [17]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count       287.000000
mean      96200.427561
std       24947.792155
min       51293.000000
25%       75670.000000
50%       90650.000000
75%      111654.000000
max      218162.440000
Name: Annual_Wage, dtype: float64


In [19]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [21]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage      Mean_Wage
0     Austin        AMD                145      77029.0   82349.641379
1     Austin      Apple                 26      87500.0   90238.038462
2     Austin  Microsoft                  6     110000.0  113333.333333
3     Austin     NVIDIA                 10      77740.0   87312.200000
4   New York      Apple                  6     111500.0  117990.666667
5   New York       Meta                 29     115000.0  122450.274138
6   New York  Microsoft                 19     120000.0  116697.341053
7   San Jose      Apple                  1     105000.0  105000.000000
8   San Jose     Micron                 15     125000.0  117683.818667
9   San Jose  Microsoft                  9     129968.0  128280.000000
10   Seattle       Meta                  4     121500.0  125750.000000
11   Seattle  Microsoft                 17     105000.0  103852.058824


In [23]:
columns_to_keep = [
    "Company",
    "LCA_CASE_EMPLOYER_NAME",
    "LCA_CASE_JOB_TITLE",
    "LCA_CASE_SOC_NAME",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2013.csv",
    index=False
)